# Data Prepocessing

Data Prepocessing adalah proses mengelola dan membersihkan data mentah agar menjadi data baru yang siap digunakan untuk analisis atau pemodelan. 

## 1. Digunakan untuk membaca data

In [27]:
import pandas as pd
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer

# ============================================
# 1. LOAD DATASET
# ============================================
df = pd.read_excel("../data/dataset_berita_200.xlsx")  # ganti ke pd.read_csv() kalau file .csv
print(df.shape)
print("Jumlah data :", len(df))
print("\nNama kolom:")
print(df.columns)

print("\nJumlah data per label:")
print(df["label"].value_counts())
df.head()

(200, 4)
Jumlah data : 200

Nama kolom:
Index(['id', 'isi_berita', 'label', 'url'], dtype='object')

Jumlah data per label:
label
sport      100
finance    100
Name: count, dtype: int64


,id,isi_berita,label,url
0,1,Chef de Mission (CdM) Indonesia Todotua Pasari...,sport,https://sport.detik.com/sport-lain/d-8655308/c...
1,2,"Banjir besar melanda Nagoya, Jepang, menjelang...",sport,https://sport.detik.com/sport-lain/d-8655303/a...
2,3,Ketua Umum Komite Olimpiade Indonesia (KOI) Ra...,sport,https://sport.detik.com/sport-lain/d-8655067/k...
3,4,Persaingan titel juara dunia semakin ketat men...,sport,https://sport.detik.com/moto-gp/d-8654776/moto...
4,5,Sektor ganda campuran kembali mengalami peromb...,sport,https://sport.detik.com/raket/d-8654666/ganda-...


## 2. Menghapus Tanda Baca Di Dalam Dokumen

* Mengubah Tipe Data (str(text)): Memastikan bahwa masukan bertipe data string (teks) untuk menghindari eror jika ada data kosong atau berupa angka.

* Case Folding (text.lower()): Mengubah semua huruf menjadi huruf kecil agar kata yang sama tidak dianggap berbeda hanya karena huruf kapital (misal: "Teks" dan "teks" dianggap sama).

* Hapus URL & Email: Menghilangkan tautan web (seperti http:// atau www.) dan alamat email menggunakan regex karena informasi ini biasanya tidak dibutuhkan dalam analisis teks.

* Hapus Angka: Menghilangkan semua karakter angka (0-9).

* Hapus Tanda Baca, Simbol, dan Emotikon: Menghapus semua karakter selain huruf alfabet dan karakter beraksen. Simbol, tanda baca, dan emoji otomatis diganti dengan spasi.

* Hapus Spasi Berlebih (.strip()): Merapikan spasi ganda akibat proses penghapusan sebelumnya menjadi satu spasi saja, sekaligus membuang spasi di awal dan akhir teks.



In [29]:
import string
import re

def hapus_tanda_baca(text):
    text = text.lower()                                 # ubah ke huruf kecil semua
    text = re.sub(r'\d+', '', text)                     # hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation))  # hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip()            # rapikan spasi ganda jadi spasi tunggal
    text = re.sub(r'https?://\S+|www\.\S+',' ',text)    # menghapus url
    text = re.sub(r'\S+@\S+',' ',text)                  # menghapus email
    return text

# Terapkan ke kolom yang diinginkan
df['isi_berita_no_punct'] = df['isi_berita'].apply(hapus_tanda_baca)

# Cek hasilnya
df[["id",'isi_berita', 'isi_berita_no_punct']].head()

,id,isi_berita,isi_berita_no_punct
0,1,Chef de Mission (CdM) Indonesia Todotua Pasari...,chef de mission cdm indonesia todotua pasaribu...
1,2,"Banjir besar melanda Nagoya, Jepang, menjelang...",banjir besar melanda nagoya jepang menjelang a...
2,3,Ketua Umum Komite Olimpiade Indonesia (KOI) Ra...,ketua umum komite olimpiade indonesia koi raja...
3,4,Persaingan titel juara dunia semakin ketat men...,persaingan titel juara dunia semakin ketat men...
4,5,Sektor ganda campuran kembali mengalami peromb...,sektor ganda campuran kembali mengalami peromb...


## 3. Normalisasi Kata Tidak Baku

Pada tahap ini dilakukan normalisasi kata, yaitu proses mengubah kata tidak baku, singkatan, atau bentuk kata yang umum digunakan dalam bahasa sehari-hari menjadi kata yang lebih baku dan seragam. Proses normalisasi menggunakan kamus normalisasi yang berisi pasangan kata tidak baku dan kata bakunya, seperti “gak” menjadi “tidak”, “yg” menjadi “yang”, “dgn” menjadi “dengan”, dan “krn” menjadi “karena”.

Sebelum proses normalisasi dilakukan, program terlebih dahulu mengidentifikasi kata-kata tidak baku yang terdapat pada setiap dokumen. Kata-kata tersebut kemudian dihitung jumlah kemunculannya menggunakan Counter. Hasil perhitungan digunakan untuk mengetahui total kata tidak baku sebelum normalisasi serta frekuensi masing-masing kata tidak baku yang ditemukan dalam seluruh dokumen. Informasi tersebut dapat digunakan sebagai dasar untuk melihat kata-kata yang perlu dinormalisasi pada tahap berikutnya.

In [4]:
from collections import Counter

kamus_normalisasi = {
     "gak": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "enggak": "tidak",
    
    "yg": "yang",
    "dgn": "dengan",
    "dr": "dari",
    "utk": "untuk",
    "tdk": "tidak",
    "tak": "tidak",
    
    "kalo": "kalau",
    "kl": "kalau",
    "klo": "kalau",
    
    "aja": "saja",
    "jgn": "jangan",
    "blm": "belum",
    "sdh": "sudah",
    "udh": "sudah",
    "dah": "sudah",
    
    "bgt": "banget",
    "banget": "sangat",
    "sm": "sama",
    "sbg": "sebagai",
    "krn": "karena",
    "karna": "karena",
    
    "tp": "tetapi",
    "tpi": "tetapi",
    "jd": "jadi",
    "jdi": "jadi",
    
    "dpt": "dapat",
    "bisa": "dapat",
    "pengen": "ingin",
    "mau": "ingin",
    
    "org": "orang",
    "orang2": "orang-orang",
    "km": "kamu",
    
    "bbrp": "beberapa",
    "sbnrnya": "sebenarnya",
    "sebenernya": "sebenarnya",
    
    "makasih": "terima kasih",
    "thx": "terima kasih",
    "thanks": "terima kasih"
}

def hitung_kata_tidak_baku(text):
    words = text.lower().split()
    ditemukan = [w for w in words if w in kamus_normalisasi]
    return ditemukan

# Total keseluruhan kata tidak baku di semua dokumen
df['kata_tidak_baku_sebelum'] = df['isi_berita_no_punct'].apply(hitung_kata_tidak_baku)
df['jumlah_kata_tidak_baku_sebelum'] = df['kata_tidak_baku_sebelum'].apply(len)

total_sebelum = df['jumlah_kata_tidak_baku_sebelum'].sum()
print(f"Total kata tidak baku SEBELUM normalisasi: {total_sebelum}")

# Rincian per kata — kata apa saja dan berapa kali muncul
semua_kata_tidak_baku = [kata for daftar in df['kata_tidak_baku_sebelum'] for kata in daftar]
rekap = Counter(semua_kata_tidak_baku)

rekap_df = pd.DataFrame(rekap.items(), columns=['kata_tidak_baku', 'frekuensi'])
rekap_df = rekap_df.sort_values('frekuensi', ascending=False).reset_index(drop=True)
rekap_df



Total kata tidak baku SEBELUM normalisasi: 402


,kata_tidak_baku,frekuensi
0,bisa,253
1,tak,60
2,nggak,27
3,mau,16
4,tp,10
5,km,8
6,gak,7
7,enggak,5
8,kl,5
9,dr,4


In [41]:
def ganti_kata_tidak_baku(text):
    words = text.lower().split()
    hasil = [kamus_normalisasi.get(w, w) for w in words]
    return " ".join(hasil)

df['isi_berita_normalized'] = df['isi_berita_no_punct'].apply(ganti_kata_tidak_baku)

df['kata_tidak_baku_sesudah'] = df['isi_berita_normalized'].apply(hitung_kata_tidak_baku)
df['jumlah_kata_tidak_baku_sesudah'] = df['kata_tidak_baku_sesudah'].apply(len)

total_sesudah = df['jumlah_kata_tidak_baku_sesudah'].sum()
print(f"Total kata tidak baku SESUDAH normalisasi: {total_sesudah}")


idx = 0  # ganti sesuai baris yang mau dilihat

# Tampilkan perbandingan sebelum dan sesudah normalisasi
df[['isi_berita_no_punct', 'isi_berita_normalized']].head(10)


Total kata tidak baku SESUDAH normalisasi: 0


,isi_berita_no_punct,isi_berita_normalized
0,chef de mission cdm indonesia todotua pasaribu...,chef de mission cdm indonesia todotua pasaribu...
1,banjir besar melanda nagoya jepang menjelang a...,banjir besar melanda nagoya jepang menjelang a...
2,ketua umum komite olimpiade indonesia koi raja...,ketua umum komite olimpiade indonesia koi raja...
3,persaingan titel juara dunia semakin ketat men...,persaingan titel juara dunia semakin ketat men...
4,sektor ganda campuran kembali mengalami peromb...,sektor ganda campuran kembali mengalami peromb...
5,pegokar muda indonesia morgan holindo tak puas...,pegokar muda indonesia morgan holindo tidak pu...
6,seri v mens world tennis championship sudah me...,seri v mens world tennis championship sudah me...
7,memperingati hari ulang tahun hut ke tni tahun...,memperingati hari ulang tahun hut ke tni tahun...
8,pacuan kuda indonesia mulai diarahkan menuju p...,pacuan kuda indonesia mulai diarahkan menuju p...
9,jelang asian games dukungan untuk atlet indone...,jelang asian games dukungan untuk atlet indone...


## 4. Mengubah Bahasa Asing menjadi Bahasa Indonesia

In [46]:
kamus_asing = {

    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",

    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}
def inggris(text):
    # Memecah teks menjadi kata-kata
    tokens = text.split()

    hasil = []

    for token in tokens:
        # Bahasa asing -> Bahasa Indonesia
        if token in kamus_asing:
            token = kamus_asing[token]

        hasil.append(token)

    # Menggabungkan kembali menjadi kalimat
    return " ".join(hasil)


df["isi_berita_final"] = df["isi_berita_normalized"].apply(inggris)

df[[
    "id",
    "isi_berita_normalized",
    "isi_berita_final"
]].head()
        

,id,isi_berita_normalized,isi_berita_final
0,1,chef de mission cdm indonesia todotua pasaribu...,chef de mission cdm indonesia todotua pasaribu...
1,2,banjir besar melanda nagoya jepang menjelang a...,banjir besar melanda nagoya jepang menjelang a...
2,3,ketua umum komite olimpiade indonesia koi raja...,ketua umum komite olimpiade indonesia koi raja...
3,4,persaingan titel juara dunia semakin ketat men...,persaingan titel juara dunia semakin ketat men...
4,5,sektor ganda campuran kembali mengalami peromb...,sektor ganda campuran kembali mengalami peromb...


## 5. Mengextrak kata Yang Ada Dalam Berita


In [47]:
# ============================================
# TOTAL KATA SEBELUM DAN SESUDAH PREPROCESSING
# ============================================

# --------------------------------------------
# 1. JUMLAH KATA SEBELUM PREPROCESSING
# --------------------------------------------

df['jumlah_kata_sebelum'] = df['isi_berita'].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

total_kata_sebelum = df['jumlah_kata_sebelum'].sum()

print("=== SEBELUM PREPROCESSING ===")
print(f"Total seluruh kata: {total_kata_sebelum}")
print(f"Rata-rata kata per berita: {df['jumlah_kata_sebelum'].mean():.2f}")
print(f"Kata terbanyak dalam satu berita: {df['jumlah_kata_sebelum'].max()}")
print(f"Kata tersedikit dalam satu berita: {df['jumlah_kata_sebelum'].min()}")


# --------------------------------------------
# 2. JUMLAH KATA SETELAH PREPROCESSING
# --------------------------------------------

df['jumlah_kata_sesudah'] = df['isi_berita_final'].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

total_kata_sesudah = df['jumlah_kata_sesudah'].sum()

print("\n=== SESUDAH PREPROCESSING ===")
print(f"Total seluruh kata: {total_kata_sesudah}")
print(f"Rata-rata kata per berita: {df['jumlah_kata_sesudah'].mean():.2f}")
print(f"Kata terbanyak dalam satu berita: {df['jumlah_kata_sesudah'].max()}")
print(f"Kata tersedikit dalam satu berita: {df['jumlah_kata_sesudah'].min()}")


# --------------------------------------------
# 3. PERBANDINGAN
# --------------------------------------------

selisih = total_kata_sebelum - total_kata_sesudah

print("\n=== PERBANDINGAN ===")
print(f"Total kata sebelum preprocessing : {total_kata_sebelum}")
print(f"Total kata sesudah preprocessing : {total_kata_sesudah}")
print(f"Selisih jumlah kata              : {selisih}")

=== SEBELUM PREPROCESSING ===
Total seluruh kata: 67113
Rata-rata kata per berita: 335.56
Kata terbanyak dalam satu berita: 1037
Kata tersedikit dalam satu berita: 126

=== SESUDAH PREPROCESSING ===
Total seluruh kata: 64444
Rata-rata kata per berita: 322.22
Kata terbanyak dalam satu berita: 1020
Kata tersedikit dalam satu berita: 115

=== PERBANDINGAN ===
Total kata sebelum preprocessing : 67113
Total kata sesudah preprocessing : 64444
Selisih jumlah kata              : 2669


In [48]:
# Jumlah kata sebelum preprocessing
df["jumlah_kata_sebelum"] = df["isi_berita"].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

# Jumlah kata setelah preprocessing
df["jumlah_kata_sesudah"] = df["isi_berita_final"].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

# Tampilkan
df[[
    "id",
    "jumlah_kata_sebelum",
    "jumlah_kata_sesudah"
]].head()

,id,jumlah_kata_sebelum,jumlah_kata_sesudah
0,1,288,280
1,2,296,287
2,3,402,392
3,4,216,210
4,5,189,186


## 6. Mengubah Label 

In [53]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

df[["id", "label", "label_num"]].head()


,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


In [54]:

print(df["label_num"].value_counts())

label_num
1    100
0    100
Name: count, dtype: int64


## 7. Melakukan Stopword Removal

Digunakan untuk mengurangi kata-kata yang kurang memberikan informasi pwenting serta mengubah kata berimbuhan menjadi bentuk dasar. Proses ini menggunakan library sastrawi untuk menghapus kata-kata umum dalam bahasa Indonesia yang tidak terlalu berpengaruh.

In [55]:
# ============================================
# 7. STOPWORD REMOVAL + STEMMING (dengan whitelist)
# ============================================
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm
tqdm.pandas()

stopword_remover = StopWordRemoverFactory().create_stop_word_remover()
stemmer = StemmerFactory().create_stemmer()

# Kata-kata yang harus dikecualikan dari stemming (nama, istilah asing, dll)
whitelist_no_stem = {
    'asian', 'games', 'championship', 'tournament', 'sport', 'sports',
    'indonesia', 'jakarta', 'jepang', 'chef', 'cdm', 'training',
    'coach', 'team', 'match', 'score', 'goal', 'championship',
    # tambahkan kata lain sesuai temuan kamu
}

def stemming_dengan_whitelist(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    hasil = []
    for w in words:
        if w.lower() in whitelist_no_stem:
            hasil.append(w)  # jangan di-stem
        else:
            hasil.append(stemmer.stem(w))  # stem kata satu per satu
    return " ".join(hasil)

df['isi_berita_no_stopword'] = df['isi_berita_final'].progress_apply(stopword_remover.remove)
df['isi_berita_clean'] = df['isi_berita_no_stopword'].progress_apply(stemming_dengan_whitelist)

df[['isi_berita_final', 'isi_berita_clean']].head(10)

100%|██████████| 200/200 [07:22<00:00,  2.21s/it]


,isi_berita_final,isi_berita_clean
0,chef de mission cdm indonesia todotua pasaribu...,chef de mission cdm indonesia todotua pasaribu...
1,banjir besar melanda nagoya jepang menjelang a...,banjir besar landa nagoya jepang jelang asian ...
2,ketua umum komite olimpiade indonesia koi raja...,ketua umum komite olimpiade indonesia koi raja...
3,persaingan titel juara dunia semakin ketat men...,saing titel juara dunia makin ketat tuju motog...
4,sektor ganda campuran kembali mengalami peromb...,sektor ganda campur alami ombak main kali deja...
5,pegokar muda indonesia morgan holindo tidak pu...,gokar muda indonesia morgan holindo puas cuma ...
6,seri v mens world tennis kejuaraan sudah memas...,seri v mens world tennis juara pasuk hari dua ...
7,memperingati hari ulang tahun hut ke tni tahun...,ingat hari ulang tahun hut tni tahun komando o...
8,pacuan kuda indonesia mulai diarahkan menuju p...,pacu kuda indonesia mulai arah tuju panggung d...
9,jelang asian pertandingan dukungan untuk atlet...,jelang asian tanding dukung atlet indonesia te...


In [56]:
# Jumlah kata sebelum preprocessing
df["jumlah_kata_sebelum"] = df["isi_berita"].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

# Jumlah kata setelah preprocessing
df["jumlah_kata_sesudah"] = df["isi_berita_clean"].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

# Tampilkan
df[[
    "id",
    "jumlah_kata_sebelum",
    "jumlah_kata_sesudah"
]].head()

,id,jumlah_kata_sebelum,jumlah_kata_sesudah
0,1,288,206
1,2,296,222
2,3,402,293
3,4,216,156
4,5,189,140


## 8. Penghitungan Kata

Digunakan untuk Menghitung kata yang ada dalam satu dataset dan kata unik yang ada dalam dataset

In [57]:
semua_kata = []

for berita in df["isi_berita_clean"]:
    semua_kata.extend(
        berita.split()
    )

kata_unik = sorted(
    set(semua_kata)
)

print(
    "Jumlah seluruh kata:",
    len(semua_kata)
)

print(
    "Jumlah kata unik:",
    len(kata_unik)
)

Jumlah seluruh kata: 50739
Jumlah kata unik: 5509


In [58]:
kata_unik[:100]

['a',
 'aadi',
 'aam',
 'aan',
 'aau',
 'abadi',
 'abalabal',
 'abang',
 'abangpalmerah',
 'abdi',
 'abdul',
 'aberdeen',
 'abraham',
 'absen',
 'abu',
 'abullah',
 'acara',
 'accurate',
 'acdacd',
 'aceh',
 'acehsumut',
 'achmad',
 'acid',
 'acosta',
 'activ',
 'activation',
 'activities',
 'acu',
 'ada',
 'adalah',
 'adaptasi',
 'adapun',
 'adhitya',
 'adi',
 'adianto',
 'adik',
 'adil',
 'adimaja',
 'adininggar',
 'administrasi',
 'administratif',
 'adna',
 'adopsi',
 'adriatik',
 'adu',
 'advokat',
 'aerodinamika',
 'aeromodelling',
 'aesi',
 'af',
 'afrianto',
 'afrika',
 'agam',
 'agama',
 'agency',
 'agenda',
 'agent',
 'agraria',
 'agreement',
 'agreementsspa',
 'agregat',
 'agresif',
 'agresivitas',
 'agung',
 'agunguniversitas',
 'agus',
 'agusman',
 'agustus',
 'ahiara',
 'ahihns',
 'ahli',
 'ahmad',
 'ahmed',
 'ahren',
 'ahsanurrohim',
 'ai',
 'aichi',
 'aichinagoya',
 'air',
 'airin',
 'airlangga',
 'airnav',
 'airport',
 'airports',
 'aja',
 'ajaib',
 'ajak',
 'ajang',
 '

In [59]:
df_kata_unik = pd.DataFrame({
    "kata_unik": kata_unik
})

df_kata_unik.head(20)

,kata_unik
0,a
1,aadi
2,aam
3,aan
4,aau
5,abadi
6,abalabal
7,abang
8,abangpalmerah
9,abdi


## Testing

* Training 160 data (80%) → digunakan untuk melatih model, sehingga model belajar mengenali pola dari data yang sudah memiliki label.

* Testing 40 data (20%) → digunakan untuk menguji model menggunakan data yang tidak digunakan saat pelatihan.

* Hasil testing kemudian digunakan untuk menghitung performa model, misalnya accuracy, precision, recall, dan F1-score.

In [61]:
from sklearn.model_selection import train_test_split

In [63]:
X_train_text, X_test_text, y_train, y_test = train_test_split(

    df["isi_berita_clean"],
    df["label_num"],

    test_size=40,

    random_state=42,

    stratify=df["label_num"]
)

In [64]:
print(
    "Jumlah training:",
    len(X_train_text)
)

print(
    "Jumlah testing:",
    len(X_test_text)
)

Jumlah training: 160
Jumlah testing: 40


In [65]:
print("TRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

TRAINING
label_num
0    80
1    80
Name: count, dtype: int64

TESTING
label_num
0    20
1    20
Name: count, dtype: int64


## 9. Melakukan TF IDF

Kode ini berfungsi untuk mengubah data teks berita menjadi format angka (vektor numerik) menggunakan metode TF-IDF (Term Frequency-Inverse Document Frequency) agar bisa dipahami dan diproses oleh algoritma machine learning.

Selain mengubah teks menjadi angka berdasarkan bobot kepentingan kata, kode ini juga secara otomatis menyaring kosa kata, yaitu membuang kata yang terlalu jarang muncul (muncul di kurang dari 2 berita) dan kata yang terlalu sering muncul (muncul di lebih dari 95% berita). Pola kosa kata ini dipelajari dari data latih (train), lalu diterapkan untuk mengubah data latih dan data uji (test) ke dalam bentuk angka.

In [71]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

In [72]:
nama_fitur = tfidf.get_feature_names_out()

print(
    "Jumlah fitur TF-IDF:",
    len(nama_fitur)
)

Jumlah fitur TF-IDF: 2355


## 10. Menampilkan matrix

Kode ini berfungsi untuk mengubah hasil perhitungan TF-IDF (yang sebelumnya berupa format matriks matematika) menjadi sebuah tabel (DataFrame) yang rapi dan mudah dibaca.

Dalam tabel ini, setiap kolom mewakili satu kosa kata yang berhasil diekstrak dari berita, dan angka-angka di dalamnya menunjukkan nilai bobot kepentingan kata tersebut pada setiap berita. Setelah tabelnya dibuat, kode akan menampilkan cuplikan 5 baris pertamanya agar Anda bisa melihat langsung hasil dari pembobotan kata-kata tersebut.

In [73]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_train_df.head()

,abdul,absen,abu,acara,acdacd,aceh,acehsumut,acosta,acu,ada,...,yogyakarta,youtube,yudhi,yusrian,yusuf,zapp,zona,zoom,zulhas,zulkifli
0,0.0,0.0,0.092565,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.026569,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.053152,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.097905,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.022178,...,0.0,0.065027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 11. Menyimpan data TF IDF

Kode ini berfungsi untuk **menyimpan tabel hasil perhitungan TF-IDF yang sudah dibuat tadi ke dalam sebuah file Excel** bernama `"data_tfidf_training.xlsx"`.

Pengaturan `index=False` digunakan untuk memastikan bahwa nomor urut baris bawaan dari sistem tidak ikut dimasukkan ke dalam file Excel tersebut, sehingga hasil akhirnya menjadi lebih rapi dan hanya berisi nama kolom/kata beserta nilai bobotnya saja.

In [79]:
tfidf_train_df.to_excel(
    "data_tfidf_training.xlsx",
    index=False
)

In [80]:
tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["label"] = y_test.reset_index(
    drop=True
)

tfidf_test_df.to_excel(
    "data_tfidf_testing.xlsx",
    index=False
)

In [82]:
import numpy as np
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

In [83]:
for kelas in sorted(
    y_train.unique()
):

    mask = (
        y_train_array == kelas
    )

    X_class = X_train_tfidf[
        mask
    ]


    # Berapa dokumen kelas tersebut
    # mengandung sebuah kata

    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )


    jumlah_dokumen_class = (
        X_class.shape[0]
    )


    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )


    class_space_density += (
        class_density
    )

In [84]:
epsilon = 1e-12

icsdf = np.log(

    (jumlah_kelas + epsilon)

    /

    (class_space_density + epsilon)
)

In [86]:
df_icsdf = pd.DataFrame({

    "kata": features,

    "ICSDF": icsdf
})

df_icsdf.sort_values(
    "ICSDF",
    ascending=False
).head(20)

,kata,ICSDF
17,advokat,4.382027
2354,zulkifli,4.382027
13,adi,4.382027
1,absen,4.382027
2353,zulhas,4.382027
2352,zoom,4.382027
2349,yusuf,4.382027
24,agus,4.382027
6,acehsumut,4.382027
2346,youtube,4.382027


## 12. TF IDF X ICSDF

In [87]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

In [89]:
from scipy.sparse import csr_matrix
X_train_icsdf = csr_matrix(
    X_train_icsdf
)

X_test_icsdf = csr_matrix(
    X_test_icsdf
)

## 13. Menentukan Kata Paling Penting

Kode ini berfungsi untuk menghitung rata-rata nilai bobot setiap kata (fitur) di seluruh data latih.

Hasil perhitungan rata-rata tersebut kemudian diubah bentuknya dan diratakan menjadi sebuah daftar angka satu dimensi (seperti sebuah baris panjang berisi nilai-nilai). Hal ini biasanya dilakukan agar skor dari tiap kata tersebut lebih mudah diproses pada tahap selanjutnya, misalnya untuk mengurutkan atau menyeleksi kata mana yang memiliki skor rata-rata paling tinggi.

In [90]:
skor_fitur = np.asarray(
    X_train_icsdf.mean(
        axis=0
    )
).ravel()

In [91]:
TOP_K = 100

In [92]:
top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

In [93]:
kata_penting = features[
    top_index
]

kata_penting[:30]

array(['balap', 'sabarreza', 'motogp', 'purbaya', 'emas', 'marc',
       'tanding', 'bandara', 'olahraga', 'asian', 'marquez', 'saham',
       'harga', 'padel', 'pupuk', 'rp', 'kuda', 'gram', 'terbang', 'lari',
       'martin', 'gim', 'aragon', 'masker', 'kapal', 'aset', 'leodaniel',
       'atlet', 'juara', 'tim'], dtype=object)

In [94]:
df_kata_penting = pd.DataFrame({

    "kata": kata_penting,

    "skor": skor_fitur[
        top_index
    ]
})

df_kata_penting.head(30)

,kata,skor
0,balap,0.078006
1,sabarreza,0.051154
2,motogp,0.051090
3,purbaya,0.050051
4,emas,0.048382
5,marc,0.047472
6,tanding,0.047431
7,bandara,0.047282
8,olahraga,0.045541
9,asian,0.044620


## 14. Reduksi Menjadi 100 Fitur ICSDF

In [95]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

In [96]:
print(
    "Sebelum seleksi:",
    X_train_tfidf.shape
)

print(
    "Sesudah ICSDF:",
    X_train_selected.shape
)

Sebelum seleksi: (160, 2355)
Sesudah ICSDF: (160, 100)


## 15. Tabel Hasil ICSDF

In [97]:
icsdf_train_df = pd.DataFrame(

    X_train_selected.toarray(),

    columns=kata_penting
)

icsdf_train_df["label"] = (

    y_train.reset_index(
        drop=True
    )
)

icsdf_train_df.head()

,balap,sabarreza,motogp,purbaya,emas,marc,tanding,bandara,olahraga,asian,...,jual,anggar,ganda,basket,rafalentino,subsidi,jonatan,ethan,bangun,label
0,0.00000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,1.347667,0.0,0.000000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0
1,0.41572,0.0,0.822333,0.0,0.000000,1.027847,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.107152,0.000000,0.0,0.0,0.0,0.0,0.0,1
2,0.00000,0.0,0.000000,0.0,1.360606,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0
3,0.00000,0.0,0.000000,0.0,0.000000,0.000000,0.194575,0.000000,0.0,0.309125,...,0.0,0.0,0.000000,0.167593,0.0,0.0,0.0,0.0,0.0,1
4,0.00000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,1


In [98]:
icsdf_train_df.to_excel(
    "data_icsdf_training.xlsx",
    index=False
)

## 16. PCA

In [100]:
from sklearn.decomposition import PCA
pca = PCA(

    n_components=20,

    random_state=42
)

In [101]:
X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

In [102]:
X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

In [103]:
print(
    X_train_pca.shape
)

print(
    X_test_pca.shape
)

(160, 20)
(40, 20)


## 17. Membuat tabel data reduksi training

In [104]:
nama_pc = [
    f"PC{i}"
    for i in range(
        1,
        21
    )
]

In [105]:
df_train_reduksi = pd.DataFrame(

    X_train_pca,

    columns=nama_pc
)

df_train_reduksi["label"] = (

    y_train.reset_index(
        drop=True
    )
)

df_train_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,-0.068208,0.018526,0.004975,-0.401990,-0.625876,-0.314082,1.101435,-0.026436,0.833593,0.028559,...,0.143770,-0.061828,-0.008970,0.031239,0.025602,0.025899,-0.079521,-0.062524,0.012631,0
1,-0.187719,-1.158015,-0.061425,1.119477,0.038866,-0.037573,0.058221,-0.136627,0.168271,-0.363347,...,0.007458,-0.001750,-0.010384,-0.042240,-0.018786,0.016786,-0.092619,-0.178711,-0.041517,1
2,-0.374568,1.119704,-0.936847,0.766422,0.053889,0.014430,0.089878,0.015929,0.076228,0.012399,...,0.017905,0.028167,0.028124,-0.034379,0.013806,-0.025154,-0.018694,0.045311,-0.000893,0
3,-0.012974,0.009693,-0.003736,-0.150301,0.005020,0.007881,0.049155,0.021585,-0.023593,0.046845,...,-0.194986,-0.066261,-0.102502,-0.044662,0.027017,0.013101,0.175393,-0.196799,-0.026074,1
4,-0.012743,-0.020682,0.002571,-0.162263,-0.109287,-0.006423,0.135317,0.007186,-0.003547,0.067601,...,-0.203039,-0.087262,-0.079374,-0.118101,0.015610,-0.029652,0.235718,0.165231,0.102481,1


## 18. Data Testing

In [106]:
df_test_reduksi = pd.DataFrame(

    X_test_pca,

    columns=nama_pc
)

df_test_reduksi["label"] = (

    y_test.reset_index(
        drop=True
    )
)

df_test_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,-0.070681,0.073682,0.023737,-0.166709,-0.070574,-0.018915,-0.031098,0.000820,-0.240595,-0.063669,...,-0.081959,-0.051881,0.132796,0.367853,-0.611096,-0.387320,-0.178926,0.031924,-0.120360,0
1,0.003033,-0.162413,-0.005589,0.030625,0.002242,0.040650,0.012198,-0.016041,-0.082641,0.197142,...,-0.017949,-0.039585,-0.047774,-0.020438,0.047619,-0.040648,0.452004,0.523223,-0.050724,1
2,0.086244,-0.005847,-0.007034,-0.114477,-0.005605,0.133547,0.017922,0.007288,-0.084149,0.073818,...,-0.379164,-0.133473,-0.245660,-0.420137,-0.023883,0.125605,-0.388089,0.283785,0.027493,1
3,-0.028956,0.004630,-0.008558,-0.110326,0.030565,-0.001160,-0.022699,0.069185,-0.038495,0.015586,...,-0.106648,-0.024196,-0.028640,0.059976,0.054934,0.010628,0.056967,-0.079878,-0.008390,1
4,-0.120756,0.194616,-0.060492,-0.112391,-0.162219,-0.049919,-0.134495,-0.079999,-0.238915,-0.121257,...,0.114140,0.053195,0.843874,-0.383564,0.378826,-0.254335,-0.152093,-0.092986,-0.079759,0


## Simpan Data Reduksi

In [107]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

## Total Training Dan Testing

In [108]:
print(
    "Training :",
    df_train_reduksi.shape
)

print(
    "Testing :",
    df_test_reduksi.shape
)

Training : (160, 21)
Testing : (40, 21)


In [109]:
df_train_reduksi.columns

Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19',
       'PC20', 'label'],
      dtype='object')